# 03 — Planning agent

**Definition:** write the whole plan **first**, execute it step by step, and re-plan only
when reality disagrees. Also called **Plan-and-Execute**.

```
        Goal
          |
          v
      +--------+
      |  PLAN  |   1 big LLM call -> ordered list of steps
      +--------+
          |
          v
      +---------+
   +->| EXECUTE |   run step[0] via a ReAct sub-agent
   |  +---------+
   |      |
   |      v
   |  +---------+
   +--| REPLAN  |   done? -> answer   |   not done? -> revise remaining steps
      +---------+
          |
          v
       Answer
```

**vs ReAct (02):** it's about *when the path is decided*.

| | ReAct | Planning |
|---|---|---|
| path decided | during execution, one step at a time | before execution, all at once |
| full task visible to LLM | only implicitly, via history | explicitly, as a list |
| human can review the plan | no | **yes** — the killer feature |
| long-horizon drift | high, it forgets the goal | low, the plan anchors it |
| adapts to surprises | instantly | only at the re-plan point |

So: planning is for when the task has many steps and the agent keeps losing the thread.

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

## Tools + the executor sub-agent

The executor is **itself a ReAct agent**. That looked like cheating the first time I saw
it, but it's the actual insight of the whole taxonomy: patterns compose.

```
Planning agent = Planner (LLM) + Executor (a ReAct agent) + Replanner (LLM)
```

The planner owns the long horizon; the ReAct executor owns the messy tool work inside one
step. Neither could do the whole job alone.

In [ ]:
from langchain_core.tools import tool


@tool
def search_flights(origin: str, destination: str) -> str:
    """Search available flights between two cities."""
    return f"Flights {origin}->{destination}: Eurowings 07:20 (EUR 89), Lufthansa 14:05 (EUR 142)"


@tool
def search_hotels(city: str, nights: int) -> str:
    """Find hotels in a city for a given number of nights."""
    return f"{city} ({nights} nights): Hotel Alpin EUR 95/night, Stadt Hotel EUR 140/night"


@tool
def get_attractions(city: str) -> str:
    """List the top tourist attractions in a city."""
    data = {
        "salzburg": "Hohensalzburg Fortress, Mirabell Gardens, Mozart's Birthplace",
        "vienna": "Schoenbrunn Palace, Stephansdom, Prater, Belvedere",
    }
    return data.get(city.lower(), f"No attraction data for {city}")


@tool
def calculate_budget(items: str) -> str:
    """Sum a comma-separated list of numeric costs, e.g. '89, 95, 95, 40'."""
    try:
        return f"Total: EUR {sum(float(x.strip()) for x in items.split(',')):.2f}"
    except Exception as e:
        return f"Error: {e}"


planning_tools = [search_flights, search_hotels, get_attractions, calculate_budget]

In [ ]:
from langchain.agents import create_agent

TOOL_MENU = "search_flights(origin, destination), search_hotels(city, nights), " \
            "get_attractions(city), calculate_budget(items)"

# A full ReAct agent, but scoped to ONE step at a time.
executor_agent = create_agent(
    model=llm,
    tools=planning_tools,
    system_prompt=(
        "You execute exactly ONE task using the available tools. Be concise and factual.\n"
        "You are running unattended: there is NO user to talk to. Never ask a clarifying "
        "question. If a detail like a date is missing, assume something sensible, say what "
        "you assumed, and call the tool anyway."
    ),
)
print("executor (ReAct sub-agent) ready")

## Schemas — and the union type that didn't survive contact

The replanner has to be able to say **two different things**:

| it says | means | graph goes to |
|---|---|---|
| here are the remaining steps | not done | `executor` |
| here is the answer | done | `END` |

The canonical plan-and-execute example models that as a Pydantic **union**, so routing is
an `isinstance` check rather than "does this prose look finished?":

```python
class Act(BaseModel):
    action: Union[Response, Plan]
```

I wrote that first and it blew up on every replan:

```
OutputParserException: Failed to parse Act from completion
{"action": {"response": {"response": "**Trip Summary & Budget** ..."}}}
  action.Response.response
    Input should be a valid string [type=string_type, input_value={'response': '...'}]
```

Read the JSON: the model wrapped the payload in a **second** layer named after the class
(`{"response": {"response": ...}}`). A union compiles to a JSON-Schema `anyOf`, and under
constrained decoding the model tags which branch it picked instead of just filling one in.
Every replan then died on a validation error.

So I flattened it into one schema with an explicit boolean discriminator. Less elegant,
but it parses every time, and `is_complete` is a far more obvious stop signal to read in
state than "which class came back".

In [ ]:
import operator
from typing import Annotated, List, Tuple, TypedDict

from pydantic import BaseModel, Field


class Act(BaseModel):
    """The replanner's decision: keep going, or answer."""

    is_complete: bool = Field(
        description="True if the objective is fully achieved and you can answer the user now."
    )
    response: str = Field(
        description="The final answer to the user. Empty string when is_complete is False."
    )
    steps: List[str] = Field(
        description="The steps that STILL need doing. Empty list when is_complete is True."
    )


class Plan(BaseModel):
    """An ordered list of steps to accomplish the goal (used by the planner node)."""

    steps: List[str] = Field(
        description="Ordered steps. Each must be self-contained and independently executable."
    )

### State — and the one key that needs a reducer

| key | reducer | why |
|---|---|---|
| `input` | none | set once, never changes |
| `plan` | none | the replanner **replaces** it wholesale |
| `past_steps` | `operator.add` | must **accumulate** across iterations |
| `response` | none | written once, at the end |

`Annotated[list, operator.add]` is the generic "append instead of overwrite" reducer.
`add_messages` from notebook 02 is just a smarter, message-aware version of the same idea.

In [ ]:
class PlanState(TypedDict):
    input: str                                                 # the original goal
    plan: List[str]                                            # remaining steps (shrinks)
    past_steps: Annotated[List[Tuple[str, str]], operator.add]  # (step, result), ACCUMULATES
    response: str                                              # final answer

## The planner node

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "For the given objective, produce a simple step-by-step plan.\n"
            f"The executor has exactly these tools: {TOOL_MENU}\n"
            "RULES:\n"
            "- Every step must be doable with those tools alone.\n"
            "- Never plan a step that asks the user for more information - nobody is there.\n"
            "- Each step must be self-contained, with all the info needed to execute it.\n"
            "- No fluff steps. Keep it to 4 steps or fewer.\n"
            "- The result of the last step must be the final answer.",
        ),
        ("user", "{input}"),
    ]
)

planner = planner_prompt | llm.with_structured_output(Plan, method="json_schema")


def plan_node(state: PlanState) -> dict:
    """ONE upfront LLM call that decomposes the whole goal."""
    plan = planner.invoke({"input": state["input"]})
    print("\nPLAN:")
    for i, s in enumerate(plan.steps, 1):
        print(f"   {i}. {s}")
    return {"plan": plan.steps}

### Look at the plan on its own first

This is the artifact that makes the whole pattern worth it — it exists, it's a list of
strings, I can print it, diff it, or hand it to a human before spending a cent on execution.

In [ ]:
draft = planner.invoke(
    {"input": "Plan a 3-night trip from Cologne to Salzburg with flights, hotel, attractions and total budget."}
)
for i, s in enumerate(draft.steps, 1):
    print(f"{i}. {s}")

## The executor node

It runs `plan[0]` only and records the result. It deliberately does **not** remove the step
from the plan — the replanner decides what's left, so there's a single source of truth.

In [ ]:
def execute_node(state: PlanState) -> dict:
    plan = state["plan"]
    task = plan[0]

    plan_text = "\n".join(f"{i}. {s}" for i, s in enumerate(plan, 1))
    prompt = f"Overall plan:\n{plan_text}\n\nYou are executing ONLY step 1: {task}"

    print(f"\nEXECUTING: {task}")
    result = executor_agent.invoke({"messages": [("user", prompt)]})
    answer = result["messages"][-1].content
    print(f"   -> {answer[:150]}")

    return {"past_steps": [(task, answer)]}    # operator.add APPENDS this tuple

## The replanner node

This is where planning earns its keep. It sees the original goal, the remaining plan, and
everything already done — then decides: revise, or finish.

**The trap:** it must return only the steps that are *still to do*. If it re-emits completed
steps you get an infinite loop. Say so explicitly in the prompt.

In [ ]:
replanner_prompt = ChatPromptTemplate.from_template(
    f"""For the given objective, come up with a simple step-by-step plan.
Each step must be self-contained and doable with these tools alone: {TOOL_MENU}
Never plan a step that asks the user for information - nobody is there to answer.
No fluff steps.
"""
    """
Your objective was:
{input}

Your original plan was:
{plan}

You have currently completed these steps:
{past_steps}

Update the plan accordingly.
ONLY include steps that STILL NEED TO BE DONE. Never repeat a completed step.
If the evidence already answers the objective, set is_complete=true and write the answer."""
)

replanner = replanner_prompt | llm.with_structured_output(Act, method="json_schema")


def replan_node(state: PlanState) -> dict:
    output = replanner.invoke(
        {
            "input": state["input"],
            "plan": "\n".join(f"- {s}" for s in state["plan"]),
            "past_steps": "\n".join(f"- {s}: {r[:200]}" for s, r in state["past_steps"]),
        }
    )

    if output.is_complete:
        print("\nREPLANNER: done")
        return {"response": output.response}

    print(f"\nREPLANNER: {len(output.steps)} step(s) remaining")
    return {"plan": output.steps}     # no reducer -> REPLACES the old plan

## Router + graph

The step cap is not decoration. My first version of this router only checked `response`,
and a run burned all 30 recursion steps: the executor kept asking for travel dates instead
of calling a tool, so the replanner never saw progress and kept inventing more steps.
`recursion_limit` did stop it — by raising `GraphRecursionError`, i.e. crashing instead of
answering. A cap I own exits deliberately and still leaves `past_steps` to inspect.

(The underlying cause was worth fixing too: neither the planner nor the executor knew what
tools existed, so the plans were unexecutable. Both prompts above now list the tool menu,
and the executor is told there is no user to ask.)

In [ ]:
from langgraph.graph import END, START, StateGraph

MAX_STEPS = 6


def should_end(state: PlanState) -> str:
    # .get() because the key may not exist at all on the first pass - plain
    # state["response"] would KeyError.
    if state.get("response"):
        return END
    if len(state["past_steps"]) >= MAX_STEPS:
        print(f"\nstep cap ({MAX_STEPS}) hit - stopping before the replanner loops forever")
        return END
    return "executor"


builder = StateGraph(PlanState)

builder.add_node("planner", plan_node)
builder.add_node("executor", execute_node)
builder.add_node("replanner", replan_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")      # always execute after planning
builder.add_edge("executor", "replanner")    # always replan after executing
builder.add_conditional_edges("replanner", should_end, ["executor", END])

plan_graph = builder.compile()
show(plan_graph)

## Run it

In [ ]:
goal = (
    "Plan a 3-night trip from Cologne to Salzburg. "
    "I need flights, a hotel, the top attractions, and the total budget."
)

final = plan_graph.invoke(
    {"input": goal, "past_steps": []},
    config={"recursion_limit": 30},    # backstop behind my own MAX_STEPS cap
)

print("\n" + "=" * 60)
print(final.get("response") or "(stopped at the step cap - no final answer written)")
print(f"\nsteps executed: {len(final['past_steps'])}")

### Watch the plan shrink

In [ ]:
for chunk in plan_graph.stream(
    {
        "input": "Find flights Cologne to Vienna and tell me the top 2 attractions there.",
        "past_steps": [],
    },
    config={"recursion_limit": 30},
    stream_mode="updates",
):
    for node, update in chunk.items():
        if "response" in update:
            print(f"[{node}] FINAL")
        elif "plan" in update:
            print(f"[{node}] plan now has {len(update['plan'])} step(s)")
        elif "past_steps" in update:
            print(f"[{node}] completed: {update['past_steps'][0][0]}")

## The payoff — human-in-the-loop on the plan

Because the plan is an explicit artifact, a human can approve or **edit** it before any
money is spent. This is the thing ReAct simply cannot do: by the time a ReAct agent has
"decided", it has already acted.

`interrupt_after=["planner"]` pauses right after the plan is produced. Interrupts need a
checkpointer — the paused state has to live somewhere.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

hitl_graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_after=["planner"],
)

config = {"configurable": {"thread_id": "trip-1"}, "recursion_limit": 30}

# 1. Run -> stops after planning
hitl_graph.invoke(
    {"input": "Find a hotel in Vienna for 2 nights and list its price.", "past_steps": []},
    config,
)

# 2. Inspect what it proposed
snapshot = hitl_graph.get_state(config)
print("\nproposed plan:")
for s in snapshot.values["plan"]:
    print("  -", s)
print("paused before:", snapshot.next)

In [ ]:
# 3. A human overrides the plan directly in state
hitl_graph.update_state(config, {"plan": ["Find hotels in Vienna for 2 nights"]})
print("plan overridden ->", hitl_graph.get_state(config).values["plan"])

# 4. Resume with input=None, meaning "continue from where you paused"
out = hitl_graph.invoke(None, config)
print("\nresult:", (out.get("response") or "(hit the step cap)")[:400])

## Notes to self

**Why the plan shrinks instead of tracking an index.** Both work, but shrinking is safer:
if the replanner reorders or inserts steps an index points at the wrong thing, "plan is
empty" is an unambiguous stop signal, and the state stays self-describing — print it and
you see exactly what's left.

**Replanner vs the critic in notebook 04.** Easy to conflate, but they judge different
things: the replanner judges the *path* ("what's left to do?"), the reflector judges the
*output* ("is this any good?"). They're orthogonal; a serious agent has both.

**Failure modes I've hit:**

| symptom | cause | fix |
|---|---|---|
| infinite loop | replanner re-emits completed steps | strengthen "ONLY remaining steps" |
| `GraphRecursionError` | replanner never sets `is_complete` | own step cap in the router, not just `recursion_limit` |
| `OutputParserException`, nested JSON | `Union[...]` schema under constrained decoding | flatten to a boolean discriminator |
| executor asks the user questions | it doesn't know it's unattended | "there is no user, assume and proceed" |
| plan steps can't be executed | planner doesn't know the tools | list the tool menu in the planner prompt |
| plan too vague | steps aren't self-contained | require each step to carry its own context |
| executor ignores the step | instruction buried in the prompt | put "execute ONLY step 1" last |
| `KeyError: 'response'` | reading a key before it's written | `state.get("response")` |

**Use it when:** long horizon, 5+ steps, drift is a real risk, or the plan needs approval.
**Skip it when:** 1–3 steps (overkill — use ReAct) or the environment is so unpredictable
that any plan is stale immediately.

**API I used:**

```python
is_complete: bool                                    # flat discriminator, not a Union
llm.with_structured_output(Act, method="json_schema")
Annotated[list, operator.add]                        # accumulate
create_agent(model=llm, tools=tools)                 # the executor
compile(checkpointer=..., interrupt_after=["planner"])
graph.update_state(config, {...})                    # human edits the plan
graph.invoke(None, config)                           # resume
```

Next: **04 — Reflective**, which adds a critic that judges the output and sends it back.